In [1]:
"""
Method 2: Structured Pruning — ResNet-50 / CIFAR-10
====================================================
L1-norm filter pruning on Conv2d layers — removes entire output filters
(structured units). Produces a sparser model without physically rebuilding
tensor shapes (PyTorch prune API zeroes filter rows).

Saves exactly 8 metrics per pruning ratio for comparison with baseline:
  accuracy, precision, recall, f1, num_parameters, model_size_mb,
  inference_time_ms, flops

Output: method2_structured_pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune_utils
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "method2_structured_pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# Fraction of output filters to remove per Conv2d layer
PRUNING_RATIOS = [0.10, 0.20, 0.30, 0.40, 0.50]
INFERENCE_RUNS = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Pruning ───────────────────────────────────────────────────────────────────
def apply_structured_pruning(model, ratio):
    """
    L1 structured pruning: remove `ratio` fraction of output filters (dim=0)
    ranked by L1 norm in each Conv2d layer.
    PyTorch zeroes the filter rows; tensor shapes are unchanged.
    """
    pruned = copy.deepcopy(model)
    for _, module in pruned.named_modules():
        if isinstance(module, nn.Conv2d) and module.weight.shape[0] > 1:
            n_to_prune = max(1, int(module.weight.shape[0] * ratio))
            n_to_prune = min(n_to_prune, module.weight.shape[0] - 1)
            prune_utils.ln_structured(module, name="weight",
                                      amount=n_to_prune, n=1, dim=0)
            prune_utils.remove(module, "weight")
    return pruned


# ── 8 Core Metrics ────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def count_parameters(model):
    """Total number of parameters (tensor shapes unchanged by structured prune)."""
    return sum(p.numel() for p in model.parameters())

def get_model_size_mb(model):
    """Actual disk size of saved .pth file in MB."""
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / (1024 ** 2)
    finally:
        os.remove(tmp)
    return size

def measure_inference_time_ms(model):
    """Average single-batch inference time in ms (GPU preferred, else CPU)."""
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32).to(DEVICE)

    if DEVICE.type == "cuda":
        for _ in range(20):
            model(dummy)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / INFERENCE_RUNS
    else:
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        return (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

def compute_flops(model):
    """FLOPs via THOP (MACs × 2). Shape-based — unchanged by zeroing filters."""
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return int(macs * 2)


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  Method 2: Structured Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device : {DEVICE}")
    print(f"  Filter pruning ratios: {[int(r*100) for r in PRUNING_RATIOS]}%")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()

    results = []

    for ratio in PRUNING_RATIOS:
        print(f"  Pruning {int(ratio*100)}% of filters per Conv2d layer...")
        pruned = apply_structured_pruning(model, ratio)

        metrics       = evaluate(pruned, loader)
        num_params    = count_parameters(pruned)
        model_size_mb = get_model_size_mb(pruned)
        inference_ms  = measure_inference_time_ms(pruned)
        flops         = compute_flops(pruned)

        row = {
            "filter_pruning_ratio": ratio,
            # ── 8 standardized metrics ──
            "accuracy"         : round(metrics["accuracy"],  6),
            "precision"        : round(metrics["precision"], 6),
            "recall"           : round(metrics["recall"],    6),
            "f1"               : round(metrics["f1"],        6),
            "num_parameters"   : num_params,
            "model_size_mb"    : round(model_size_mb, 4),
            "inference_time_ms": round(inference_ms, 4),
            "flops"            : flops,
        }
        results.append(row)

        print(f"    Acc: {metrics['accuracy']:.4f} | Params: {num_params:,} | "
              f"Size: {model_size_mb:.2f} MB | Inf: {inference_ms:.2f} ms | "
              f"FLOPs: {flops/1e9:.3f} G")

    report = {
        "method"     : "structured_pruning",
        "description": "L1 structured filter pruning — entire Conv2d output filters zeroed (dim=0)",
        "baseline"   : baseline,
        "results"    : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()


  Method 2: Structured Pruning — ResNet-50 / CIFAR-10
  Device : cuda
  Filter pruning ratios: [10, 20, 30, 40, 50]%

  Pruning 10% of filters per Conv2d layer...
    Acc: 0.9322 | Params: 23,520,842 | Size: 90.03 MB | Inf: 104.59 ms | FLOPs: 2.623 G
  Pruning 20% of filters per Conv2d layer...
    Acc: 0.9310 | Params: 23,520,842 | Size: 90.03 MB | Inf: 103.58 ms | FLOPs: 2.623 G
  Pruning 30% of filters per Conv2d layer...
    Acc: 0.9313 | Params: 23,520,842 | Size: 90.03 MB | Inf: 143.87 ms | FLOPs: 2.623 G
  Pruning 40% of filters per Conv2d layer...
    Acc: 0.9227 | Params: 23,520,842 | Size: 90.03 MB | Inf: 183.85 ms | FLOPs: 2.623 G
  Pruning 50% of filters per Conv2d layer...
    Acc: 0.8734 | Params: 23,520,842 | Size: 90.03 MB | Inf: 178.44 ms | FLOPs: 2.623 G

  ✓ Saved → method2_structured_pruning.json
